# ⏱️ Aula 12 — Uniforme e exponencial na operação de servidores

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados da área de infraestrutura** de uma empresa de software: a equipe que cuida dos servidores. Agora a variável de interesse não é mais uma **contagem** (0, 1, 2, ...), e sim um **tempo**, que pode assumir qualquer valor num intervalo. Por isso usamos os **modelos contínuos** da Aula 12: a **distribuição uniforme** e a **distribuição exponencial**.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**.

Lembre da aula: num modelo contínuo, probabilidade é **área sob a curva da densidade**. Para os dois modelos de hoje, essa área tem fórmula pronta:

- **Uniforme** em $[\alpha, \beta]$: $P(a \le X \le b) = \dfrac{b-a}{\beta-\alpha}$ (a área de um retângulo);
- **Exponencial** com parâmetro $\lambda$: $F(x) = P(X \le x) = 1 - e^{-\lambda x}$, para $x \ge 0$.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula da aula diz.

In [ ]:
import numpy as np                    # calculadora de números e sorteios
import matplotlib.pyplot as plt       # gráficos

rng = np.random.default_rng(42)       # sorteador, semente 42
print("Tudo pronto!")

## 1. A hora do backup (distribuição uniforme)

### A situação
Todo dia, o backup dos servidores começa num momento sorteado **entre 2h00 e 3h00** da madrugada. O sorteio é proposital: se todos os servidores começassem no mesmo minuto, a rede ficaria congestionada. Qualquer minuto dentro dessa hora é **igualmente provável**.

### A pergunta
Seja $X$ = minutos depois das 2h00 em que o backup começa ($0 \le X \le 60$).
1. Qual a chance de o backup começar **entre 2h10 e 2h25**?
2. Um relatório pesado roda nos **primeiros 6 minutos** e nos **últimos 12 minutos** da hora. Qual a chance de o backup começar num desses momentos de conflito?
3. Em média, a que horas o backup começa? E qual o desvio padrão?

### A ferramenta da aula
"Igualmente provável em todo o intervalo" é a **uniforme contínua**: $X \sim U[0, 60]$, com densidade constante $f(x) = \frac{1}{60}$.

### A resposta pela conta
1. $P(10 \le X \le 25) = \dfrac{25 - 10}{60} = \dfrac{15}{60} = 0{,}25$;
2. $P(X \le 6 \text{ ou } X \ge 48) = \dfrac{6}{60} + \dfrac{12}{60} = \dfrac{18}{60} = 0{,}30$ (os dois pedaços não se sobrepõem, então as áreas somam);
3. $E(X) = \dfrac{0 + 60}{2} = 30$ minutos (2h30) e $\text{Var}(X) = \dfrac{(60 - 0)^2}{12} = 300$, ou seja, desvio padrão $\sqrt{300} \approx 17{,}3$ minutos.

### O que o código faz
1. **simula 100.000 noites**: em cada uma, `rng.uniform(0, 60)` sorteia o minuto de início do backup;
2. conta em que fração das noites aconteceu cada evento e compara com as contas;
3. desenha o **histograma** dos horários sorteados (barras que mostram quantas noites caíram em cada faixa de minutos).

In [ ]:
alfa, beta = 0, 60                                     # intervalo [0, 60] minutos
inicio = rng.uniform(alfa, beta, size=100_000)         # 100.000 noites simuladas

entre_10_25 = (inicio >= 10) & (inicio <= 25)
conflito = (inicio <= 6) | (inicio >= 48)              # "|" quer dizer "ou"

print(f"P(10 <= X <= 25)     teórico: {15 / 60:.4f}   empírico: {entre_10_25.mean():.4f}")
print(f"P(conflito)          teórico: {18 / 60:.4f}   empírico: {conflito.mean():.4f}")
print(f"Média (minutos)      teórico: {(alfa + beta) / 2:.2f}   empírico: {inicio.mean():.2f}")
print(f"Desvio padrão (min)  teórico: {np.sqrt((beta - alfa)**2 / 12):.2f}   empírico: {inicio.std():.2f}")

plt.hist(inicio, bins=30, density=True, color="steelblue", edgecolor="black")
plt.axhline(1 / 60, color="firebrick", linestyle="--", label="densidade teórica f(x) = 1/60")
plt.xlabel("Minutos depois das 2h00")
plt.ylabel("Densidade")
plt.title("Início do backup: U[0, 60]")
plt.legend()
plt.show()

**Como ler:**
- A simulação confirma as contas: em cerca de **25%** das noites o backup começa entre 2h10 e 2h25, e em cerca de **30%** delas cai num momento de conflito com o relatório.
- O histograma é **"achatado"**: todas as barras têm mais ou menos a mesma altura, na linha vermelha $1/60$. Esse é o retrato da uniforme — nenhum minuto é privilegiado.

**Conclusão para a infraestrutura:** 30% de chance de conflito por noite é muito. Uma solução simples é sortear o início só entre 2h06 e 2h48, longe do relatório.

## 2. Quanto tempo dura um disco? (distribuição exponencial)

### A situação
Os servidores usam discos SSD. Pelo histórico da empresa, o **tempo até a falha** de um disco tem média de **500 dias**. O fornecedor oferece **garantia de 50 dias**: se o disco falhar antes disso, é trocado de graça.

### A pergunta
Seja $X$ = tempo (em dias) até a falha de um disco.
1. Qual a chance de um disco falhar **dentro da garantia**?
2. Qual a chance de um disco durar **entre 200 e 2.000 dias**?
3. O fornecedor quer trocar **no máximo 5%** dos discos. Qual deve ser o prazo da garantia?

### A ferramenta da aula
Tempo até a ocorrência de um evento (a falha), com taxa constante: **exponencial**. Como $E(X) = \frac{1}{\lambda} = 500$, o parâmetro é $\lambda = \frac{1}{500} = 0{,}002$ falha por dia. É o mesmo raciocínio do exemplo das lâmpadas da aula.

### A resposta pela conta
1. $P(X < 50) = F(50) = 1 - e^{-50/500} = 1 - e^{-0{,}1} \approx 0{,}0952$;
2. $P(200 < X < 2000) = F(2000) - F(200) = e^{-0{,}4} - e^{-4} \approx 0{,}6703 - 0{,}0183 = 0{,}6520$;
3. queremos $t$ com $F(t) = 1 - e^{-t/500} = 0{,}05$, ou seja, $e^{-t/500} = 0{,}95$, $t = -500 \cdot \ln(0{,}95) \approx 25{,}6$ dias.

### O que o código faz
1. escreve a **função de distribuição** $F(x) = 1 - e^{-\lambda x}$ como uma função do Python;
2. calcula as três respostas;
3. **simula a vida de 100.000 discos** (`rng.exponential` sorteia tempos com média 500) e compara;
4. desenha o histograma dos tempos simulados com a curva da densidade $f(x) = \lambda e^{-\lambda x}$ por cima.

In [ ]:
media = 500                                            # dias, em média, até a falha
lam = 1 / media                                        # λ = 0,002 falha por dia


def F(x):
    # função de distribuição da exponencial: P(X <= x)
    return 1 - np.exp(-lam * x)


garantia_ideal = -media * np.log(0.95)                 # prazo com 5% de trocas

discos = rng.exponential(media, size=100_000)          # vida de 100.000 discos simulados

print(f"P(X < 50)            teórico: {F(50):.4f}   empírico: {(discos < 50).mean():.4f}")
print(f"P(200 < X < 2000)    teórico: {F(2000) - F(200):.4f}   empírico: {((discos > 200) & (discos < 2000)).mean():.4f}")
print(f"Garantia para 5% de trocas: {garantia_ideal:.1f} dias")
print(f"Média (dias)         teórico: {media}       empírico: {discos.mean():.1f}")

x = np.linspace(0, 2500, 300)
plt.hist(discos, bins=60, range=(0, 2500), density=True, color="steelblue", edgecolor="black", alpha=0.7)
plt.plot(x, lam * np.exp(-lam * x), color="firebrick", linewidth=2, label="densidade f(x) = λ e^(-λx)")
plt.axvline(50, color="black", linestyle="--", label="fim da garantia (50 dias)")
plt.xlabel("Dias até a falha")
plt.ylabel("Densidade")
plt.title("Tempo até a falha de um disco: exponencial com média de 500 dias")
plt.legend()
plt.show()

**Como ler:**
- Cerca de **9,5%** dos discos falham dentro da garantia de 50 dias — quase 1 em cada 10.
- Cerca de **65%** dos discos duram entre 200 e 2.000 dias.
- Para trocar no máximo 5% dos discos, a garantia teria de cair para cerca de **25 dias**.
- O gráfico mostra o formato típico da exponencial: a densidade é **maior perto do zero** e cai continuamente. Falhas cedo são as mais prováveis "por dia", e uma minoria de discos dura muito mais do que a média.

Um detalhe que surpreende: a média é 500 dias, mas **mais da metade** dos discos falha antes disso ($F(500) = 1 - e^{-1} \approx 0{,}63$). A cauda longa à direita "puxa" a média para cima.

## 3. A ponte com a Aula 11: o tempo entre chamados

### A situação e a pergunta
Volte à central de suporte da Aula 11, que recebe em média **4 chamados por hora** (Poisson com $\lambda = 4$). Agora a pergunta muda de "**quantos** chamados chegam?" para "**quanto tempo** passa entre um chamado e o próximo?".
1. Qual o tempo médio entre dois chamados?
2. Qual a chance de passarem **mais de 15 minutos** sem chamado?

### A ferramenta da aula
Quando o **número** de ocorrências num intervalo é Poisson com taxa $\lambda$, o **tempo entre** duas ocorrências é **exponencial** com o mesmo $\lambda$. Medindo o tempo em horas: $T \sim \text{Exp}(\lambda = 4)$.

### A resposta pela conta
1. $E(T) = \dfrac{1}{4}$ de hora $= 15$ minutos;
2. $P(T > 0{,}25) = 1 - F(0{,}25) = e^{-4 \times 0{,}25} = e^{-1} \approx 0{,}3679$ — exatamente a chance de "nenhum chamado em 15 minutos" calculada na Aula 11.

### O que o código faz
Monta uma central de suporte "de mentira": sorteia **100.000 intervalos** entre chamados com a exponencial, empilha esses intervalos um depois do outro para obter os horários de chegada e, então, **conta quantos chamados caíram em cada hora**. Se a ponte estiver certa, essas contagens devem se comportar como uma Poisson com média 4.

In [ ]:
intervalos = rng.exponential(1 / 4, size=100_000)      # tempo entre chamados, em horas (média 1/4)
chegadas = np.cumsum(intervalos)                       # horário de cada chamado (soma acumulada)

print(f"Tempo médio entre chamados: {intervalos.mean() * 60:.1f} minutos (teórico: 15)")
print(f"P(T > 15 min)   teórico: {np.exp(-1):.4f}   empírico: {(intervalos > 0.25).mean():.4f}")

horas_completas = int(chegadas[-1])                    # quantas horas inteiras foram simuladas
por_hora = np.bincount(chegadas.astype(int))[:horas_completas]   # chamados em cada hora
print(f"Chamados por hora: média {por_hora.mean():.2f} e variância {por_hora.var():.2f} (Poisson: as duas = 4)")

**Como ler:** sorteando só os **tempos entre chamados** (exponencial), a contagem de chamados **por hora** saiu com média e variância perto de **4** — o comportamento da Poisson da Aula 11. E a chance de passar mais de 15 minutos sem chamado é cerca de **37%** pelas duas aulas.

São duas maneiras de olhar o mesmo fenômeno: a **Poisson conta** as ocorrências, a **exponencial mede o tempo** entre elas.

## Para fechar

Neste notebook, o time de infraestrutura usou os dois modelos contínuos da Aula 12:

- **Uniforme** $U[\alpha, \beta]$: quando todo valor do intervalo é igualmente provável — a hora sorteada do backup. Probabilidade = comprimento do pedaço ÷ comprimento do intervalo;
- **Exponencial**: tempo até um evento — a falha de um disco, a chegada do próximo chamado. Probabilidade pela função $F(x) = 1 - e^{-\lambda x}$;
- **A ponte com a Poisson:** se as ocorrências são Poisson($\lambda$), o tempo entre elas é exponencial($\lambda$).

Na **Aula 13** vem o modelo contínuo mais importante da Estatística: a **distribuição normal**, com sua curva em forma de sino.